In [1]:
import cv2
import numpy as np
import time
import threading
from datetime import datetime
import csv
import os
import pickle
from sklearn.neighbors import KNeighborsClassifier
from ultralytics import YOLO   # pip install ultralytics


# ─────────────────────────────────────────────────────────────────────
# Threaded camera reader
# ─────────────────────────────────────────────────────────────────────
class CameraStream:
    def __init__(self, src=0):
        self.cap = cv2.VideoCapture(src)
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1920)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
        self.cap.set(cv2.CAP_PROP_FPS, 30)
        self.ret, self.frame = self.cap.read()
        self._lock   = threading.Lock()
        self._stop   = False
        self._thread = threading.Thread(target=self._update, daemon=True)
        self._thread.start()

    def _update(self):
        while not self._stop:
            ret, frame = self.cap.read()
            with self._lock:
                self.ret, self.frame = ret, frame

    def read(self):
        with self._lock:
            return self.ret, self.frame.copy() if self.ret else (False, None)

    def isOpened(self):  return self.cap.isOpened()
    def get(self, prop): return self.cap.get(prop)

    def release(self):
        self._stop = True
        self._thread.join()
        self.cap.release()


# ─────────────────────────────────────────────────────────────────────
class DeskTracker:

    INFER_WIDTH         = 640   # width for YOLO inference
    FACE_RECOG_INTERVAL = 10    # face-id every N frames
    LOG_PRESENCE_DELAY  = 2.0   # seconds before a detection is logged

    def __init__(self):

        # ── YOLOv8 ───────────────────────────────────────────────────
        self.model = YOLO('Yolo_model/yolov8n.pt')

        self.DESK_RELATED_CLASSES   = {56, 60, 63, 64, 65}
        self.PERSON_CLASS_ID        = 0
        self.confidence_threshold   = 0.3
        self.person_conf            = 0.50
        self.face_recognition_threshold = 18

        # ── Face recognition ─────────────────────────────────────────
        try:
            with open('data/Organizations.pkl', 'rb') as f:
                self.ORGANIZATIONS = pickle.load(f)
            with open('data/names.pkl', 'rb') as f:
                self.NAMES = pickle.load(f)
            with open('data/Emails.pkl', 'rb') as f:
                self.EMAILS = pickle.load(f)
            with open('data/EmployeeIDs.pkl', 'rb') as f:
                self.EmployeeIDs = pickle.load(f)
            with open('data/faces_data.pkl', 'rb') as f:
                self.FACES = pickle.load(f) / 255.0
                print(f"FACES shape: {self.FACES.shape}")
            self.knn = KNeighborsClassifier(n_neighbors=5)
            self.knn.fit(self.FACES, self.EmployeeIDs)
            print("Face recognition loaded successfully.")
            self.face_recognition_enabled = True
        except Exception as e:
            print(f"Face model error: {e}. Using 'Unknown' for all.")
            self.NAMES = []; self.EmployeeIDs = []
            self.ORGANIZATIONS = []; self.EMAILS = []
            self.face_recognition_enabled = False

        self.face_cascade = cv2.CascadeClassifier(
            'data/haarcascade_frontalface_default.xml')

        # ── Desk ROI ─────────────────────────────────────────────────
        self.DESK_ROI_PT1 = None
        self.DESK_ROI_PT2 = None
        self.desk_mask    = None

        # ── Tracking ─────────────────────────────────────────────────
        self.trackers           = {}
        self.person_status      = {}
        self.MAX_CENTROIDS      = 10
        self.DISAPPEARANCE_TIME = 2.0

        self.employee_absence_start = None
        self.visitor_absence_start  = None
        self.last_employee_present  = None
        self.last_visitor_present   = None

        # ── 1-second presence timer before logging ────────────────────
        # { name -> {'location': str, 'since': float, 'logged': bool, 'empID': str} }
        self._pending_log = {}

        # ── Face cache ───────────────────────────────────────────────
        self._face_cache     = {}
        self._face_frame_ctr = 0

        # ── Logging ──────────────────────────────────────────────────
        date = datetime.now().strftime("%d-%m-%Y")
        os.makedirs('Logs', exist_ok=True)
        log_folder = os.path.join('Logs', f'Log_{date}')
        os.makedirs(log_folder, exist_ok=True)

        # detection_log now includes a "Present Duration (s)" column
        self.log_file = os.path.join(log_folder, f'detection_log_{date}.csv')
        if not os.path.exists(self.log_file):
            with open(self.log_file, 'w', newline='') as f:
                csv.writer(f).writerow(
                    ['Date & Time', 'EmployeeID', 'Name',
                     'Detection Status', 'Present Duration (s)'])

        self.absence_log_file = os.path.join(
            log_folder, f'absence_log_{date}.csv')
        if not os.path.exists(self.absence_log_file):
            with open(self.absence_log_file, 'w', newline='') as f:
                csv.writer(f).writerow(
                    ['Date & Time', 'EmployeeID', 'Name', 'Absence Duration (s)'])

        self.person_absence_start  = {}
        self.person_absence_logged = {}

        self.monitor_area   = None
        self.display_width  = 1280
        self.display_height = 720
        self.debug_mode     = False

    # ─────────────────────────────────────────────────────────────────
    # LOGGING HELPERS
    # ─────────────────────────────────────────────────────────────────
    def log_event(self, msg):
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

    def log_detection(self, name, employee_id, status, present_duration=0.0):
        ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.log_file, 'a', newline='') as f:
            csv.writer(f).writerow(
                [ts, employee_id, name, status, f"{present_duration:.1f}"])
        self.log_event(
            f"[DETECTION] {name} ({employee_id}) -> {status} "
            f"[confirmed after {present_duration:.1f}s]")

    def log_absence(self, name, employee_id, dur):
        if dur <= 3:
            return
        ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.absence_log_file, 'a', newline='') as f:
            csv.writer(f).writerow([ts, employee_id, name, f"{dur:.1f}"])
        self.log_event(
            f"[ABSENCE]   {name} ({employee_id}) absent for {dur:.1f}s")

    # ─────────────────────────────────────────────────────────────────
    # 1-SECOND PRESENCE TIMER
    # ─────────────────────────────────────────────────────────────────
    def _check_and_log(self, name, employee_id, location):

        now   = time.time()
        entry = self._pending_log.get(name)

        if entry is None or entry['location'] != location:
            # New detection or location changed — reset timer
            self._pending_log[name] = {
                'location': location,
                'since':    now,
                'logged':   False,
                'empID':    employee_id,
            }
            return   # not logged yet — keep waiting

        # Same location: check elapsed time
        elapsed = now - entry['since']
        if not entry['logged'] and elapsed >= self.LOG_PRESENCE_DELAY:
            entry['logged'] = True
            self.log_detection(name, employee_id,
                               f"Detected - {location}", elapsed)
            self.person_status[name] = location


    # ─────────────────────────────────────────────────────────────────
    # ABSENCE TRACKING
    # ─────────────────────────────────────────────────────────────────
    def mark_person_present(self, name, employee_id):
        if name in self.person_absence_start:
            dur = time.time() - self.person_absence_start.pop(name)
            if name != "People":
                self.log_absence(name, employee_id, dur)

    def mark_person_absent(self, name):
        if name not in self.person_absence_start:
            self.person_absence_start[name] = time.time()

    # ─────────────────────────────────────────────────────────────────
    # DESK AREA DETECTION
    # ─────────────────────────────────────────────────────────────────
    def detect_desk_area(self, stream):
        self.log_event("Detecting desk area with YOLOv8 ...")
        desk_candidates = []
        frame_count = 0
        frame = None

        while frame_count < 5:
            ret, frame = stream.read()
            if not ret or frame is None or frame.size == 0:
                time.sleep(0.05)
                continue

            height, width = frame.shape[:2]
            small = cv2.resize(frame, (self.INFER_WIDTH,
                                       int(height * self.INFER_WIDTH / width)))
            sx = width  / small.shape[1]
            sy = height / small.shape[0]

            results = self.model(small, conf=self.confidence_threshold,
                                 verbose=False)
            for box in results[0].boxes:
                cls_id = int(box.cls[0])
                if cls_id in self.DESK_RELATED_CLASSES:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    desk_candidates.append(
                        [int(x1*sx), int(y1*sy),
                         int((x2-x1)*sx), int((y2-y1)*sy),
                         float(box.conf[0]), cls_id])
            frame_count += 1

        if len(desk_candidates) < 2:
            self.log_event("Not enough desk objects – using default area.")
            if frame is not None:
                height, width = frame.shape[:2]
            else:
                height, width = 480, 640
            min_x = int(width  * 0.2); min_y = int(height * 0.4)
            max_x = int(width  * 0.8); max_y = int(height * 0.9)
        else:
            x_c  = [d[0]      for d in desk_candidates]
            y_c  = [d[1]      for d in desk_candidates]
            mx_c = [d[0]+d[2] for d in desk_candidates]
            my_c = [d[1]+d[3] for d in desk_candidates]
            min_x = max(0,      int(min(x_c)  - 0.1*width))
            min_y = max(0,      int(min(y_c)  - 0.1*height))
            max_x = min(width,  int(max(mx_c) + 0.1*width))
            max_y = min(height, int(max(my_c) + 0.1*height))
            self.log_event(f"Desk area: ({min_x},{min_y}) -> ({max_x},{max_y})")

        self.DESK_ROI_PT1 = (min_x, min_y)
        self.DESK_ROI_PT2 = (max_x, max_y)
        self.monitor_area = (min_x, min_y, max_x, max_y)
        return self.monitor_area

    # ─────────────────────────────────────────────────────────────────
    # FACE RECOGNITION
    # ─────────────────────────────────────────────────────────────────
    def get_name_from_face(self, face_region):
        if not self.face_recognition_enabled or face_region.size == 0:
            return "People", "N/A", ""
        if face_region.shape[0] < 30 or face_region.shape[1] < 30:
            return "People", "N/A", ""

        gray  = cv2.cvtColor(face_region, cv2.COLOR_BGR2GRAY)
        faces = []
        for scale, neighbors, min_size in [
            (1.1,  5, (20, 20)),
            (1.05, 4, (25, 25)),
            (1.2,  5, (30, 30)),
        ]:
            detected = self.face_cascade.detectMultiScale(
                gray, scale, neighbors, minSize=min_size)
            if len(detected) > 0:
                faces = detected
                break

        Name, EmployeeID, Email = "People", "N/A", ""
        for (x, y, wf, hf) in faces:
            crop = face_region[y:y+hf, x:x+wf]
            if crop.size == 0:
                continue
            try:
                resized = cv2.resize(crop, (75, 100))
                g       = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
                flat    = g.reshape(1, -1) / 255.0
                if flat.shape[1] != self.FACES.shape[1]:
                    continue
                dists, _ = self.knn.kneighbors(flat)
                if dists[0][0] <= self.face_recognition_threshold:
                    emp_id = self.knn.predict(flat)[0]
                    if emp_id in self.EmployeeIDs:
                        idx        = self.EmployeeIDs.index(emp_id)
                        Name       = self.NAMES[idx]
                        EmployeeID = self.EmployeeIDs[idx] if idx < len(self.EmployeeIDs) else "N/A"
                        Email      = self.EMAILS[idx]      if idx < len(self.EMAILS)      else ""
                        break
            except Exception:
                continue
        return Name, EmployeeID, Email

    def get_name_from_centroid(self, cx, cy, frame):
        cache_key = (cx // 64, cy // 64)
        if self._face_frame_ctr % self.FACE_RECOG_INTERVAL != 0:
            return self._face_cache.get(cache_key, ("People", "N/A", ""))

        h, w = frame.shape[:2]
        search_regions = [
            (max(0,cx-180), max(0,cy-280), min(w,cx+180), min(h,cy+50)),
            (max(0,cx-220), max(0,cy-320), min(w,cx+220), min(h,cy+80)),
            (max(0,cx-150), max(0,cy-350), min(w,cx+150), min(h,cy)),
            (max(0,cx-200), max(0,cy-200), min(w,cx+200), min(h,cy+150)),
        ]
        for idx, (rx1, ry1, rx2, ry2) in enumerate(search_regions):
            if self.debug_mode:
                color = [(255,0,255),(0,255,255),(255,255,0),(255,128,0)][idx]
                cv2.rectangle(frame, (rx1,ry1), (rx2,ry2), color, 2)
                cv2.putText(frame, f"Search {idx+1}", (rx1, ry1-5),
                            cv2.FONT_HERSHEY_COMPLEX, 0.4, color, 1)
            result = self.get_name_from_face(frame[ry1:ry2, rx1:rx2])
            if result[0] != "People":
                self._face_cache[cache_key] = result
                return result

        result = ("People", "N/A", "")
        self._face_cache[cache_key] = result
        return result

    # ─────────────────────────────────────────────────────────────────
    # MAIN RUN
    # ─────────────────────────────────────────────────────────────────
    def run(self):
        stream = CameraStream(0)
        if not stream.isOpened():
            self.log_event("Error: Cannot open camera.")
            return
        self.log_event(
            f"Camera opened - "
            f"{int(stream.get(cv2.CAP_PROP_FRAME_WIDTH))}x"
            f"{int(stream.get(cv2.CAP_PROP_FRAME_HEIGHT))}")
        self.detect_desk_area(stream)
        self.log_event("Tracker running... Press 'q' to quit.")
        self._run_full_mode(stream)

    # ─────────────────────────────────────────────────────────────────
    # FULL MODE
    # ─────────────────────────────────────────────────────────────────
    def _run_full_mode(self, stream):
        last_cleanup = time.time()
        fps_t0       = time.time()
        fps_counter  = 0
        fps_display  = 0.0

        cv2.namedWindow('Employee Desk Tracking', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Employee Desk Tracking',
                         self.display_width, self.display_height)

        while True:
            ret, frame = stream.read()
            if not ret or frame is None or frame.size == 0:
                time.sleep(0.01)
                continue

            h, w = frame.shape[:2]
            self._face_frame_ctr += 1

            # FPS
            fps_counter += 1
            if (time.time() - fps_t0) >= 1.0:
                fps_display = fps_counter / (time.time() - fps_t0)
                fps_counter = 0
                fps_t0      = time.time()

            # Desk mask
            if self.desk_mask is None or self.desk_mask.shape[:2] != (h, w):
                self.desk_mask = np.zeros((h, w), np.uint8)
                if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                    cv2.rectangle(self.desk_mask,
                                  self.DESK_ROI_PT1, self.DESK_ROI_PT2, 255, -1)

            # YOLO on downscaled frame
            scale_w = self.INFER_WIDTH / w
            small   = cv2.resize(frame, (self.INFER_WIDTH, int(h * scale_w)))
            results = self.model(small, conf=self.person_conf,
                                 classes=[self.PERSON_CLASS_ID], verbose=False)

            current_in_desk   = {}
            current_outside   = {}
            employee_detected = False
            visitor_detected  = False
            seen_names        = set()

            for box in results[0].boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                x1 = int(x1 / scale_w); y1 = int(y1 / scale_w)
                x2 = int(x2 / scale_w); y2 = int(y2 / scale_w)
                conf = float(box.conf[0])
                cx, cy = (x1+x2)//2, (y1+y2)//2

                name, EmployeeID, email = self.get_name_from_centroid(cx, cy, frame)
                seen_names.add(name)

                if name not in self.trackers:
                    self.trackers[name] = {
                        'centroids': [], 'last_seen': 0,
                        'EmployeeID': EmployeeID, 'email': email}
                self.trackers[name].update({
                    'bbox': (x1,y1,x2,y2), 'last_seen': time.time(),
                    'EmployeeID': EmployeeID, 'email': email})
                self.trackers[name]['centroids'].append((cx, cy))
                if len(self.trackers[name]['centroids']) > self.MAX_CENTROIDS:
                    self.trackers[name]['centroids'].pop(0)

                # Desk overlap
                overlap_ratio = 0
                if x2 > x1 and y2 > y1:
                    ys  = slice(max(0,y1), min(h,y2))
                    xs  = slice(max(0,x1), min(w,x2))
                    roi = self.desk_mask[ys, xs]
                    if roi.size > 0:
                        overlap_ratio = np.sum(roi > 0) / roi.size

                location = 'Inside Desk' if overlap_ratio > 0.4 else 'Outside Desk'

                if 'Inside' in location:
                    current_in_desk[name] = (EmployeeID, email)
                    if name != "People":
                        employee_detected = True
                        self.last_employee_present = time.time()
                        self.employee_absence_start = None
                else:
                    current_outside[name] = (EmployeeID, email)
                    visitor_detected = True
                    self.last_visitor_present = time.time()
                    self.visitor_absence_start = None

                # ── 1-SECOND PRESENCE TIMER ──────────────────────────
                self._check_and_log(name, EmployeeID, location)
                # ────────────────────────────────────────────────────

                self.mark_person_present(name, EmployeeID)

                # Draw bbox
                color = (0,255,0) if 'Inside' in location else (0,0,255)
                cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
                role  = "Employee" if 'Inside' in location else "Visitor"
                label = f"{name} - {role} ({location.split()[0]})"
                lsz   = cv2.getTextSize(label, cv2.FONT_HERSHEY_COMPLEX, 0.8, 2)[0]
                cv2.rectangle(frame, (x1,y1-45), (x1+lsz[0]+15, y1), color, -1)
                cv2.putText(frame, label, (x1+8, y1-15),
                            cv2.FONT_HERSHEY_COMPLEX, 0.8, (255,255,255), 2)
                id_display = (EmployeeID if EmployeeID and EmployeeID != "N/A"
                              else ("Employee" if 'Inside' in location else "Visitor"))
                cv2.putText(frame, f"{id_display} | Conf: {conf:.2f}",
                            (x1, y2+25), cv2.FONT_HERSHEY_COMPLEX, 0.6, color, 2)



            # Mark absent for unseen people
            for tracked_name in list(self.person_status.keys()):
                if tracked_name not in seen_names:
                    self.mark_person_absent(tracked_name)

            if not employee_detected and self.employee_absence_start is None:
                self.employee_absence_start = time.time()
            if not visitor_detected and self.visitor_absence_start is None:
                self.visitor_absence_start = time.time()


            # Draw desk ROI
            if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                cv2.rectangle(frame,
                              self.DESK_ROI_PT1, self.DESK_ROI_PT2,
                              (255,255,0), 4)
                ms = 40
                p1, p2 = self.DESK_ROI_PT1, self.DESK_ROI_PT2
                for pt, dx, dy in [
                    (p1,            (ms,0),  (0,ms)),
                    ((p2[0],p1[1]),(-ms,0),  (0,ms)),
                    ((p1[0],p2[1]),(ms,0),   (0,-ms)),
                    (p2,           (-ms,0),  (0,-ms)),
                ]:
                    cv2.line(frame,pt,(pt[0]+dx[0],pt[1]+dx[1]),(0,255,255),6)
                    cv2.line(frame,pt,(pt[0]+dy[0],pt[1]+dy[1]),(0,255,255),6)

            # HUD
            fps_label = f"FPS: {fps_display:.1f}"
            fz = cv2.getTextSize(fps_label, cv2.FONT_HERSHEY_COMPLEX, 0.7, 2)[0]
            cv2.putText(frame, fps_label, (w - fz[0] - 15, 35),
                        cv2.FONT_HERSHEY_COMPLEX, 0.7, (0,255,255), 2)

            emp_text  = ("Employee Status: PRESENT"
                         if employee_detected else "Employee Status: ABSENT")
            emp_color = (0,255,0) if employee_detected else (0,0,255)
            cv2.putText(frame, emp_text, (15,35),
                        cv2.FONT_HERSHEY_COMPLEX, 0.7, emp_color, 2)

            if not employee_detected and self.employee_absence_start:
                dur = time.time() - self.employee_absence_start
                cv2.putText(frame, f"Employee Absence: {dur:.1f}s",
                            (15,65), cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,0,255), 2)

            vis_count = len(current_outside)
            vis_text  = (f"Visitor Status: {vis_count} PRESENT"
                         if visitor_detected else "Visitor Status: NONE")
            vis_color = (0,165,255) if visitor_detected else (128,128,128)
            cv2.putText(frame, vis_text, (15,100),
                        cv2.FONT_HERSHEY_COMPLEX, 0.7, vis_color, 2)

            cv2.putText(frame,
                        f"Inside: {len(current_in_desk)} | "
                        f"Outside: {len(current_outside)} | "
                        f"Total: {len(self.trackers)}",
                        (15,135), cv2.FONT_HERSHEY_COMPLEX, 0.7, (255,255,0), 2)

            if current_in_desk:
                nin = ", ".join(list(current_in_desk.keys())[:3])
                if len(current_in_desk) > 3: nin += f" +{len(current_in_desk)-3}"
                cv2.putText(frame, f"Inside Desk: {nin}", (15,165),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,255,0), 2)
            if current_outside:
                nout = ", ".join(list(current_outside.keys())[:3])
                if len(current_outside) > 3: nout += f" +{len(current_outside)-3}"
                cv2.putText(frame, f"Outside Desk: {nout}", (15,195),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,165,255), 2)

            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            cv2.putText(frame, f"Time: {ts}", (15,225),
                        cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,255,255), 2)
            cv2.putText(frame, "Press q to Quit", (15,250),
                        cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,200,0), 2)

            cv2.imshow('Employee Desk Tracking', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        stream.release()
        cv2.destroyAllWindows()
        self.log_event(
            f"Logs saved -> {self.log_file}  |  {self.absence_log_file}")


# ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import sys

    tracker = DeskTracker()
    for arg in sys.argv[1:]:
        if arg.lower() in ['debug', 'd', '--debug']:
            tracker.debug_mode = True
            print("DEBUG MODE – face search regions visualised")

    print("\n" + "="*60)
    print("Starting Employee Desk Tracker  [YOLOv8 - optimised]")
    print("="*60)
    print("\nPress 'q' to quit\n" + "="*60 + "\n")

    tracker.run()

FACES shape: (50, 7500)
Face recognition loaded successfully.

Starting Employee Desk Tracker  [YOLOv8 - optimised]

Press 'q' to quit



[19:18:14] Camera opened - 1280x720
[19:18:14] Detecting desk area with YOLOv8 ...


[19:18:14] Not enough desk objects – using default area.
[19:18:14] Tracker running... Press 'q' to quit.


[19:18:17] [DETECTION] People (N/A) -> Detected - Inside Desk [confirmed after 2.5s]


[19:18:18] [DETECTION] Piyush (23bcs061) -> Detected - Inside Desk [confirmed after 2.4s]


[19:18:33] Logs saved -> Logs\Log_03-03-2026\detection_log_03-03-2026.csv  |  Logs\Log_03-03-2026\absence_log_03-03-2026.csv
